# AutoShield enforcer walkthrough

This notebook runs the real `enforcer.py` validation and **dry-run** paths. It writes audit output only into a temporary directory and does not install, remove, or change firewall rules.

Run cells from top to bottom. The sample IPs are public examples; change them to try other candidates.

> **Safety:** This notebook intentionally never uses `--apply`. Live firewall testing requires an authorized, disposable Linux VM, UFW, the reviewed root-owned helper, and the narrow sudoers configuration from the README. Do not enable live changes on a system where losing network access would be harmful.

Jupyter already runs an asyncio event loop, while `enforcer.main()` calls `asyncio.run()`. Therefore this walkthrough starts `enforcer.py` as a separate Python process instead of calling `enforcer.main()` in a notebook cell.

## 1. Set up the walkthrough

Keep this notebook in the AutoShield project folder beside `enforcer.py`. A temporary directory keeps demonstration state and audit output away from normal project files.

In [32]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

import enforcer

project_dir = Path(enforcer.__file__).resolve().parent
demo_dir = Path(tempfile.mkdtemp(prefix="autoshield-notebook-"))
state_file = demo_dir / "blocks.sqlite3"
audit_file = demo_dir / "audit.jsonl"

print(f"Project directory: {project_dir}")
print(f"Temporary test directory: {demo_dir}")
print("No live firewall changes will be made.")

Project directory: C:\Users\Jesse\Documents\Auto-Shield
Temporary test directory: C:\Users\Jesse\AppData\Local\Temp\autoshield-notebook-v_7xm_8w
No live firewall changes will be made.


## 2. Choose candidate addresses

Private, loopback, link-local, and other non-global addresses should be rejected.

In [33]:
candidate_ips = ["8.8.8.8", "1.1.1.1"]
candidate_ips

['8.8.8.8', '1.1.1.1']

## 3. Validate and normalize

`validate_candidates` parses addresses with Python's `ipaddress` module, rejects non-public addresses, normalizes input, and removes duplicates. An invalid address rejects the complete batch.

In [34]:
validated_ips = enforcer.validate_candidates(candidate_ips)
print(f"Validated {len(validated_ips)} unique public address(es):")
validated_ips

Validated 2 unique public address(es):


['8.8.8.8', '1.1.1.1']

## 4. Run the enforcer in dry-run mode

This launches the actual CLI as a child process with `--dry-run`. It exercises CLI validation, expiry calculation, and audit logging without creating block state or invoking the privileged UFW helper.

In [35]:
arguments = [
    *candidate_ips,
    "--dry-run",
    "--ttl",
    "30m",
    "--state-file",
    str(state_file),
    "--audit-file",
    str(audit_file),
]

completed = subprocess.run(
    [sys.executable, str(project_dir / "enforcer.py"), *arguments],
    check=False,
    capture_output=True,
    text=True,
    timeout=30,
)
print(completed.stdout, end="")
if completed.stderr:
    print(completed.stderr, end="")
print(f"Enforcer exit code: {completed.returncode}")
assert completed.returncode == 0, "Dry-run failed; inspect the output above."

{"event":"block_preview","event_id":"a0fbfb43-80fb-4c67-8dfa-24aaa584839b","expires_at":"2026-10-04T01:10:21.329195+00:00","ip_address":"8.8.8.8","status":"dry_run","timestamp":"2026-10-04T00:40:21.329234+00:00"}
{"event":"block_preview","event_id":"61afe693-c6fa-4270-9240-a72f382e05ce","expires_at":"2026-10-04T01:10:21.329195+00:00","ip_address":"1.1.1.1","status":"dry_run","timestamp":"2026-10-04T00:40:21.329276+00:00"}
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Enforcer exit code: 0


## 5. Inspect the JSONL audit records

Each line is an independent JSON event. Confirm the events report `dry_run` and that no SQLite block-state file was created.

In [36]:
audit_events = [
    json.loads(line)
    for line in audit_file.read_text(encoding="utf-8").splitlines()
]

for event in audit_events:
    print(json.dumps(event, indent=2))

assert all(event["status"] == "dry_run" for event in audit_events)
assert not state_file.exists(), "Dry-run unexpectedly created block state."
print("\nVerified: audit entries were written and no block state was created.")

{
  "event": "block_preview",
  "event_id": "a0fbfb43-80fb-4c67-8dfa-24aaa584839b",
  "expires_at": "2026-10-04T01:10:21.329195+00:00",
  "ip_address": "8.8.8.8",
  "status": "dry_run",
  "timestamp": "2026-10-04T00:40:21.329234+00:00"
}
{
  "event": "block_preview",
  "event_id": "61afe693-c6fa-4270-9240-a72f382e05ce",
  "expires_at": "2026-10-04T01:10:21.329195+00:00",
  "ip_address": "1.1.1.1",
  "status": "dry_run",
  "timestamp": "2026-10-04T00:40:21.329276+00:00"
}

Verified: audit entries were written and no block state was created.


## 6. Preview expiry cleanup

This read-only scan uses the isolated state path. Since no leases were created, it reports that no state database exists.

In [37]:
expiry_result = subprocess.run(
    [
        sys.executable,
        str(project_dir / "enforcer.py"),
        "--expire",
        "--dry-run",
        "--state-file",
        str(state_file),
        "--audit-file",
        str(audit_file),
    ],
    check=False,
    capture_output=True,
    text=True,
    timeout=30,
)
print(expiry_result.stdout, end="")
if expiry_result.stderr:
    print(expiry_result.stderr, end="")
assert expiry_result.returncode == 0
assert not state_file.exists()
print("Expiry preview completed without firewall or state changes.")

{"event":"expiry_scan","event_id":"2786d154-aa0c-42ce-b6fd-251414ca0b22","status":"no_state","timestamp":"2026-10-04T00:40:42.413808+00:00"}
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Expiry preview completed without firewall or state changes.


## Next steps

- Change a candidate to a private or malformed address to observe fail-closed validation.
- Review the audit records and configured 30-minute preview lease.
- Actual firewall integration testing is a separate step: follow the Linux VM checklist in `README.md`, review the exact rule, and ensure console recovery access before using the explicit `--apply` option outside this notebook.
- A successful walkthrough validates dry-run only. It does not prove UFW, sudoers, GPG, or systemd expiry are configured correctly.